# Overture MK2 Closure Sizing
Edited 10/20/2026, Joris Lindinger

Based on OVMK1 master document, Closure V2 technical documentation and Nozzle End V3.2 closure calculations

Closure layout:
- Mounted to IRIS with 3 evenly distributed U-brackets on the injector-side flange
- Mounted to injector with bolts (TBD)
- Nozzle held in by a bolted thrust ring with 16 radial bolts (replaces the snap ring)

## General data

Thin-walled assumption valid, since diameter/thickness = 32 > 20. Thick-walled formulas are still used below since they are exact

Closure loads (injector side, nozzle end) are multiplied by the load multiplier K_d to account for the dynamic startup shock, the rest of the analysis is static

In [ ]:
import math
import numpy as np

# geometry data
d_inner = 4 #inches, matched to nozzle outer diameter from DXF
t_wall = 0.1875 #inches
d_outer = d_inner + 2*t_wall #inches
l = 12 #inches

# material data (304 stainless) STANDARD TEMPERATURE FOR NOW, CHECK AT HIGH T AGAIN!!!
sigma_yield = 29732.7 #yield strength, psi
tau_yield = 0.577*sigma_yield #shear yield strength based on van mises, psi
nu = 0.29 #poisson's ratio
E = 28e6 #young's modulus, psi

# other data
p_chamber = 300 #psi
p_atm = 14.7 #psi, sea level assumed for now
F_thrust = 1100 #lbf (4.9 kN)
K_d = 2.0 #load multiplier for dynamic startup shock
n_safety = 2 #safety factor 2, from OVMK1 documentation

## Hoop stress

Thick-walled formula

r =^ radius at point of interest<br>
A, B =^ boundary conditions

P_a =^ internal pressure<br>
P_b =^ external pressure<br>
a =^ inner radius<br>
b =^ outer radius<br>

In [ ]:
P_a = p_chamber
P_b = p_atm

a = d_inner/2
b = a + t_wall

r = a #using inner radius for worst-case estimate

A = (P_a*a**2 - P_b*b**2)/(b**2-a**2)
B = a**2*b**2*(P_a-P_b)/(b**2 - a**2)


sigma_hoop = A + B / r**2

print(f"Hoop stress: {sigma_hoop} psi")

# Axial stress

This was somehow not included in the OVMK1 master document

Is uniformly distributed across the wall, so formula is simple

In [18]:
sigma_axial = (P_a*a**2-P_b*b**2)/(b**2-a**2)
print(f"Axial stress: {sigma_axial} psi")

Axial stress: 1438.7686567164178 psi


## Radial stress

A, B, r same as for hoop stress

In [19]:
sigma_rad = A - B / r**2

print(f"Radial stress: {sigma_rad} psi")

Radial stress: -300.0 psi


# Von Mises Stress

Source: (https://en.wikipedia.org/wiki/Von_Mises_yield_criterion), State of stress: Principal stresses

In [20]:
sigma_van_mises = math.sqrt(0.5*((sigma_hoop-sigma_axial)**2+(sigma_axial-sigma_rad)**2+(sigma_rad-sigma_hoop)**2))

print(f"Applied van mises stress: {sigma_van_mises} psi")
print(f"Yield strength: {sigma_yield} psi")
print(f"Safety factor: {sigma_yield/sigma_van_mises}")
if sigma_yield/sigma_van_mises < n_safety:
    print("FAIL!")

Applied van mises stress: 3011.635656041123 psi
Yield strength: 29732.7 psi
Safety factor: 9.872608574134244


# Closure loads

Chamber pressure acts on the full bore area and pushes the injector out of the chamber. The load on the nozzle end is smaller, since only the converging section of the nozzle gets pushed out, see Nozzle end load

$F_{eject} = P_c \frac{\pi D_{bore}^2}{4}$<br>
$F_{dyn} = K_d F_{eject}$

In [21]:
F_eject = p_chamber*math.pi*d_inner**2/4 #lbf
F_dyn = F_eject*K_d #lbf

print(f"Ejection force: {F_eject} lbf")
print(f"Dynamic ejection force: {F_dyn} lbf")

Ejection force: 3769.9111843077517 lbf
Dynamic ejection force: 7539.822368615503 lbf


# Injector side

Flange welded to the chamber, mounted to IRIS with 3 U-brackets

Assumptions:
- Materials stay in the elastic region
- Chamber is modeled as a rigid wall, so the flange acts like a cantilevered plate
- Bracket load disperses into the flange at 45°
- Bolt stress dissipates radially
- Weld is modeled as a fillet with an isosceles right triangle cross section

## Flange bending

Each bracket carries a third of the dynamic thrust at the bracket hole radius, cantilevered from the chamber

r_bracket =^ radius of the bracket holes<br>
L =^ lever arm, measured from the bore radius (conservative)<br>
w_bracket =^ bracket width<br>
b_eff =^ effective flange width at 45° dispersion<br>
b_max =^ maximum flange width per bracket (bore circumference / 3)<br>
t_flange =^ flange thickness

$\sigma = \frac{Mc}{I} = \frac{6M}{b t_{flange}^2}$, with $b = \min(b_{eff}, b_{max})$

In [22]:
r_bracket = 3.131 #inches
w_bracket = 2.5 #inches
t_flange = 5/16 #inches
n_brackets = 3

F_bracket = F_thrust*K_d/n_brackets #lbf
L = r_bracket - d_inner/2 #inches
M_flange = F_bracket*L #in-lbf

b_eff = w_bracket + 2*L*math.tan(math.radians(45)) #inches
b_max = math.pi*d_inner/n_brackets #inches
b_flange = min(b_eff, b_max) #inches

sigma_flange = 6*M_flange/(b_flange*t_flange**2)
t_flange_min = math.sqrt(6*M_flange*n_safety/(sigma_yield*b_flange)) #inches

print(f"Flange width used: {b_flange} in")
print(f"Applied bending stress: {sigma_flange} psi")
print(f"Yield strength: {sigma_yield} psi")
print(f"Safety factor: {sigma_yield/sigma_flange}")
print(f"Minimum flange thickness: {t_flange_min} in")
if sigma_yield/sigma_flange < n_safety:
    print("FAIL!")

Flange width used: 4.1887902047863905 in
Applied bending stress: 12165.406599206523 psi
Yield strength: 29732.7 psi
Safety factor: 2.4440366836517646
Minimum flange thickness: 0.28269046449752533 in


## Weld shear

Fillet weld around the chamber outer diameter carries the dynamic ejection force on the injector

t_weld =^ fillet weld leg size<br>
t_throat =^ weld throat thickness<br>
A_weld =^ weld throat area

$t_{throat} = t_{weld} \cos 45° = \frac{t_{weld}}{\sqrt{2}}$<br>
$\tau_{weld} = \frac{F_{dyn}}{A_{weld}} = \frac{\sqrt{2} F_{dyn}}{\pi D_{o} t_{weld}}$

In [23]:
t_weld = 1/8 #inches

t_throat = t_weld*math.cos(math.radians(45)) #inches
A_weld = math.pi*d_outer*t_throat #in^2
tau_weld = F_dyn/A_weld
t_weld_min = math.sqrt(2)*F_dyn*n_safety/(math.pi*d_outer*tau_yield) #inches

print(f"Applied shear stress: {tau_weld} psi")
print(f"Yield shear strength: {tau_yield} psi")
print(f"Safety factor: {tau_yield/tau_weld}")
print(f"Minimum weld leg size: {t_weld_min} in")
if tau_yield/tau_weld < n_safety:
    print("FAIL!")

Applied shear stress: 6206.377233728782 psi
Yield shear strength: 17155.7679 psi
Safety factor: 2.7642161044878737
Minimum weld leg size: 0.09044155397044021 in


# Nozzle end

Bolted thrust ring (the internal plate in V3.2, replaces the snap ring). The ring sits in the chamber bore behind the nozzle and is held by radial bolts through the chamber wall

Source: Nozzle End V3.2 closure calculations technical documentation, plus the checks from the earlier version

Assumptions:
- Radial symmetry
- Linear elasticity: with all safety factors >= 2, everything stays below 50% of yield
- Nozzle load: the converging section of the nozzle is fully at chamber pressure, the diverging section fully at atmospheric pressure
- Load is shared evenly across all bolts
- Bolts are cantilevered from the chamber wall and loaded along their thread engagement in the ring (conservative, a tight fit between ring and chamber takes out most of the bending)
- Tear-out is pure shear along two planes per bolt
- Saint-Venant's principle: stress concentrations at the drilled hole edges dissipate radially. The material is ductile, so it can yield locally and redistribute them (https://www.mechref.org/mf/Stress_Concentrations/?origin=coursemenu)
- The crush seal deforms plastically and absorbs the thermal expansion of the thrust ring
- Thrust ring is 304 stainless, same as the chamber

n_bolts =^ number of radial bolts<br>
d_bolt =^ bolt major diameter<br>
n_threads =^ threads per inch<br>
e =^ edge distance from hole center to end of chamber<br>
t_p =^ thrust ring thickness (axial)<br>
r_i =^ thrust ring inner radius, slightly bigger than the nozzle exit radius<br>
r_o =^ thrust ring outer radius (bore radius)<br>
l_engage =^ thread engagement length in the ring<br>
e_plate =^ edge distance from hole center to the ring face (axial)<br>
w_seal =^ contact width of the crush seal on the ring<br>
r_entrance, r_throat, r_exit =^ nozzle internal radii at the entrance, throat and exit

In [ ]:
n_bolts = 16
d_bolt = 0.25 #inches
n_threads = 20 #threads per inch, 1/4-20 UNC assumed
e = 0.25 #inches
t_p = 0.5 #inches
r_i = 1.75 #inches
r_o = d_inner/2 #inches
l_engage = r_o - r_i #inches, assumed to be the full radial width of the ring
e_plate = 0.25 #inches
w_seal = 0.25 #inches, back-calculated from the V3.2 lift result, CHECK AGAINST THE SEAL!!!

# nozzle internal contour (V3.2)
r_entrance = 1.985 #inches
r_throat = 0.888 #inches
r_exit = 1.697 #inches

# bolt material: alloy steel socket head screws, ASTM A574 (https://www.mcmaster.com/91251A535/)
sigma_yield_bolt = 153000 #yield strength, psi

## Nozzle end load

Chamber pressure pushes on the converging section of the nozzle, out of the chamber. The diverging section is assumed to be at atmospheric pressure, which is conservative since the real pressure there is higher and pushes the nozzle back in. The throat area is subtracted from both sections since it's the transition between them. Atmospheric pressure also acts on the outside of the nozzle's aft face. Altogether this is just the gauge pressure on the converging area

A_c =^ converging area<br>
A_d =^ diverging area

$A = \pi r^2$ for the entrance, throat and exit<br>
$A_c = A_{entrance} - A_{throat}$<br>
$A_d = A_{exit} - A_{throat}$<br>
$F_{conv} = P_c A_c$<br>
$F_{div} = P_a A_d$<br>
$F_{atm} = P_a (A_{entrance} - A_{exit})$<br>
$F_{eject,noz} = F_{conv} - F_{atm} - F_{div} = (P_c - P_a) A_c$<br>
$F_{dyn,noz} = K_d F_{eject,noz}$

In [ ]:
A_entrance = math.pi*r_entrance**2 #in^2
A_throat = math.pi*r_throat**2 #in^2
A_exit = math.pi*r_exit**2 #in^2
A_conv = A_entrance - A_throat #in^2
A_div = A_exit - A_throat #in^2

F_conv = p_chamber*A_conv #lbf
F_div = p_atm*A_div #lbf
F_atm = p_atm*(A_entrance - A_exit) #lbf
F_eject_nozzle = F_conv - F_atm - F_div #lbf
F_dyn_nozzle = F_eject_nozzle*K_d #lbf

print(f"Nozzle ejection force: {F_eject_nozzle} lbf")
print(f"Dynamic nozzle ejection force: {F_dyn_nozzle} lbf")

## Radial bolt shear and bending

V3.2 failure mode 1. The ring loads each bolt along its thread engagement, so the bolt is a cantilever from the chamber wall with a uniform distributed load. The worst point is the shear plane between the ring OD and the chamber ID, where the bending moment is largest and the full shear acts. The bolt is in plane stress there, which is combined with von Mises

The ring is tapped right at the interface, so the threads cross the shear plane and the minor diameter is used for both the shear area and the bending section

Note: V3.2 uses the major diameter and the plate's axial length t_p as the loaded length. The bolts are radial, so the loaded length is the thread engagement in the ring (its radial width)

F_bolt =^ load per bolt<br>
w_bolt =^ distributed load along the thread engagement<br>
c =^ distance from the neutral axis<br>
I_bolt =^ moment of inertia of the bolt section

$d_{minor} = d_{bolt} - \frac{1.299038}{n_{threads}}$ (Shigley Table 8-2)<br>
$F_{bolt} = \frac{F_{dyn,noz}}{n}$<br>
$\tau_{bolt} = \frac{F_{bolt}}{\pi d_{minor}^2/4}$<br>
$w_{bolt} = \frac{F_{bolt}}{l_{engage}}$, $M_{max} = \frac{w_{bolt} l_{engage}^2}{2} = \frac{F_{bolt} l_{engage}}{2}$<br>
$\sigma_{bend} = \frac{M_{max} c}{I_{bolt}}$, with $c = \frac{d_{minor}}{2}$ and $I_{bolt} = \frac{\pi d_{minor}^4}{64}$<br>
$\sigma_v = \sqrt{\sigma_{bend}^2 + 3\tau_{bolt}^2}$

In [ ]:
d_minor = d_bolt - 1.299038/n_threads #inches, minor diameter (Shigley Table 8-2)
A_bolt = math.pi*d_minor**2/4 #in^2, minor diameter area
I_bolt = math.pi*d_minor**4/64 #in^4
c_bolt = d_minor/2 #inches

F_bolt = F_dyn_nozzle/n_bolts #lbf
tau_bolt = F_bolt/A_bolt

w_bolt = F_bolt/l_engage #lbf/in, distributed along the thread engagement
M_bolt = w_bolt*l_engage**2/2 #in-lbf, at the shear plane
sigma_bend_bolt = M_bolt*c_bolt/I_bolt

sigma_v_bolt = math.sqrt(sigma_bend_bolt**2 + 3*tau_bolt**2)

print(f"Load per bolt: {F_bolt} lbf")
print(f"Applied shear stress: {tau_bolt} psi")
print(f"Applied bending stress: {sigma_bend_bolt} psi")
print(f"Applied von mises stress: {sigma_v_bolt} psi")
print(f"Yield strength: {sigma_yield_bolt} psi")
print(f"Safety factor: {sigma_yield_bolt/sigma_v_bolt}")
if sigma_yield_bolt/sigma_v_bolt < n_safety:
    print("FAIL!")

## Chamber bearing

Bolts crush the holes in the chamber wall into ovals

$\sigma_{br,chamber} = \frac{F_{dyn,noz}}{n d_{bolt} t_{wall}}$

In [ ]:
A_br_chamber = n_bolts*d_bolt*t_wall #in^2
sigma_br_chamber = F_dyn_nozzle/A_br_chamber

print(f"Applied bearing stress: {sigma_br_chamber} psi")
print(f"Yield strength: {sigma_yield} psi")
print(f"Safety factor: {sigma_yield/sigma_br_chamber}")
if sigma_yield/sigma_br_chamber < n_safety:
    print("FAIL!")

## Thrust ring bearing

Bolts crush the holes in the thrust ring into ovals. Since the bolts are radial, the bearing length is the thread engagement in the ring

$\sigma_{br,ring} = \frac{F_{dyn,noz}}{n d_{bolt} l_{engage}}$

In [ ]:
A_br_ring = n_bolts*d_bolt*l_engage #in^2
sigma_br_ring = F_dyn_nozzle/A_br_ring

print(f"Applied bearing stress: {sigma_br_ring} psi")
print(f"Yield strength: {sigma_yield} psi")
print(f"Safety factor: {sigma_yield/sigma_br_ring}")
if sigma_yield/sigma_br_ring < n_safety:
    print("FAIL!")

## Chamber edge tear-out

Bolts rip through the lip between the holes and the end of the chamber, two shear planes per bolt

$\tau_{tear} = \frac{F_{dyn,noz}}{2 n e t_{wall}}$

In [ ]:
A_tear = 2*n_bolts*e*t_wall #in^2
tau_tear = F_dyn_nozzle/A_tear

print(f"Applied shear stress: {tau_tear} psi")
print(f"Yield shear strength: {tau_yield} psi")
print(f"Safety factor: {tau_yield/tau_tear}")
if tau_yield/tau_tear < n_safety:
    print("FAIL!")

## Thrust ring tear-out

V3.2 failure mode 2. The bolts stay in place and rip through the ring between the holes and the ring face, in pure shear along two planes per bolt. Each plane runs from the hole center to the ring face (e_plate) and through the depth of the tapped hole (l_engage)

Note: V3.2 uses the axial length t_p as the depth of the planes, but the holes are radial, so their depth is the radial width of the ring

$A_{tear,ring} = 2 e_{plate} l_{engage}$ (per bolt)<br>
$\tau_{tear,ring} = \frac{F_{bolt}}{A_{tear,ring}}$

In [ ]:
A_tear_ring = 2*e_plate*l_engage #in^2, per bolt
tau_tear_ring = F_bolt/A_tear_ring

print(f"Applied shear stress: {tau_tear_ring} psi")
print(f"Yield shear strength: {tau_yield} psi")
print(f"Safety factor: {tau_yield/tau_tear_ring}")
if tau_yield/tau_tear_ring < n_safety:
    print("FAIL!")

## Thrust ring bending

Ring modeled as an annular plate, outer edge simply supported by the bolts, inner edge free, with the full dynamic load as a uniform pressure over the whole ring face (nozzle bears on the entire annulus). Max stress is the tangential stress at the inner edge

Note: The notion V2 document used Roark's formula for a *solid* circular plate with an annular line load, which ignores the hole

Source: Roark's Formulas for Stress and Strain, 7th ed. (Young & Budynas), Table 11.2, case 2a (annular plate, uniform pressure, outer edge simply supported, inner edge free). Roark's notation: a = r_o, b = r_i, and the load starts at the inner edge (Roark's load radius r_o = b), so it covers the whole face. Checked against the tabulated coefficients for a plate with a central hole, uniform load, outer edge simply supported (sigma_max = k q r_o^2/t^2, k = 0.592 at r_o/r_i = 1.25 for nu = 0.3; Timoshenko & Woinowsky-Krieger, Theory of Plates and Shells, Table 3; https://roymech.org/Useful_Tables/Mechanics/Plates). The table starts at r_o/r_i = 1.25, so the closed form is used for this ring (r_o/r_i = 1.14)

q =^ uniform pressure on the ring face<br>
C_7, L_17 =^ Roark plate constants<br>
M_tb =^ tangential bending moment per unit length at the inner edge

$q = \frac{F_{dyn,noz}}{\pi (r_o^2 - r_i^2)}$<br>
$C_7 = \frac{1}{2}(1-\nu^2)\left(\frac{r_o}{r_i} - \frac{r_i}{r_o}\right)$<br>
$L_{17} = \frac{1}{4}\left[1 - \frac{1-\nu}{4}\left(1 - \left(\frac{r_i}{r_o}\right)^4\right) - \left(\frac{r_i}{r_o}\right)^2\left(1 + (1+\nu)\ln\frac{r_o}{r_i}\right)\right]$<br>
$M_{tb} = \frac{\theta_b D (1-\nu^2)}{r_i}$ with $\theta_b = \frac{q r_o^3 L_{17}}{D C_7}$, so $M_{tb} = \frac{q r_o^3 (1-\nu^2) L_{17}}{r_i C_7}$<br>
$\sigma_{max} = \frac{6 M_{tb}}{t_p^2}$


In [ ]:
q_ring = F_dyn_nozzle/(math.pi*(r_o**2 - r_i**2)) #psi, uniform pressure over the ring face
C_7 = 0.5*(1 - nu**2)*(r_o/r_i - r_i/r_o) #Roark Table 11.2
L_17 = 0.25*(1 - (1 - nu)/4*(1 - (r_i/r_o)**4) - (r_i/r_o)**2*(1 + (1 + nu)*math.log(r_o/r_i))) #Roark Table 11.2, load from r_i to r_o
M_tb = q_ring*r_o**3*(1 - nu**2)*L_17/(r_i*C_7) #in-lbf/in, tangential moment at the inner edge

sigma_ring = 6*M_tb/t_p**2
t_p_min = t_p*math.sqrt(n_safety*sigma_ring/sigma_yield) #inches, stress scales with 1/t_p^2

print(f"Ring face pressure: {q_ring} psi")
print(f"Applied bending stress: {sigma_ring} psi")
print(f"Yield strength: {sigma_yield} psi")
print(f"Safety factor: {sigma_yield/sigma_ring}")
print(f"Minimum ring thickness: {t_p_min} in")
if sigma_yield/sigma_ring < n_safety:
    print("FAIL!")

## Thrust ring bending, thick ring

The annular plate formula above assumes a thin plate (thickness much smaller than the radial width), but here t_p = 0.5 in is twice the radial width of 0.25 in. A thick, narrow ring doesn't bend like a plate: each cross section rotates as a rigid body about its centroid (ring rotation), which stretches the ring circumferentially on one face and compresses it on the other. This only produces hoop stress, which is largest at the inner edge on the two faces

Same load and support as the plate case: uniform pressure over the ring face, reacted at the outer edge by the bolts

Source: Timoshenko, Strength of Materials, Part II, 3rd ed., circular ring of rectangular cross section under uniformly distributed twisting couples. The hoop strain from the rotation varies with 1/r across the width, which gives the ln term. For r_o - r_i << r_i it reduces to the straight beam result $\sigma = \frac{6M_{ring}}{(r_o - r_i)t_p^2}$. For a narrow ring the plate solution converges to this one (the radial moments vanish), so the two should agree closely

r_load =^ centroid radius of the pressure load<br>
M_ring =^ bending moment on each ring cross section (twisting moment per radian of ring)<br>
phi =^ rotation of the cross section<br>
y =^ axial distance from the ring mid-plane

$r_{load} = \frac{2}{3}\frac{r_o^3 - r_i^3}{r_o^2 - r_i^2}$<br>
$M_{ring} = \frac{F_{dyn,noz}(r_o - r_{load})}{2\pi}$<br>
$\sigma(r, y) = \frac{E \varphi y}{r}$, with equilibrium $\int \sigma y \, dA = M_{ring}$ giving $E\varphi = \frac{12 M_{ring}}{t_p^3 \ln(r_o/r_i)}$<br>
$\sigma_{max} = \frac{6 M_{ring}}{t_p^2 \, r_i \ln(r_o/r_i)}$


In [ ]:
r_load = 2/3*(r_o**3 - r_i**3)/(r_o**2 - r_i**2) #inches, centroid radius of the uniform pressure
M_ring = F_dyn_nozzle*(r_o - r_load)/(2*math.pi) #in-lbf, moment on each ring cross section

sigma_ring_thick = 6*M_ring/(t_p**2*r_i*math.log(r_o/r_i))
t_p_min_thick = t_p*math.sqrt(n_safety*sigma_ring_thick/sigma_yield) #inches, stress scales with 1/t_p^2

print(f"Load centroid radius: {r_load} in")
print(f"Applied bending stress: {sigma_ring_thick} psi")
print(f"Plate formula / thick ring formula: {sigma_ring/sigma_ring_thick}")
print(f"Yield strength: {sigma_yield} psi")
print(f"Safety factor: {sigma_yield/sigma_ring_thick}")
print(f"Minimum ring thickness: {t_p_min_thick} in")
if sigma_yield/sigma_ring_thick < n_safety:
    print("FAIL!")

## Thrust ring rotation (seal lift)

V3.2 angular deformation. The bolts stop the ring from moving axially at the outer edge, but it can still rotate, which lifts the inner edge off the crush seal. If the seal can't follow, gas leaks between the ring and the chamber. Worst case: the full nozzle load as a line load on the inner edge. The plate formulas are fine here since the deflection is far below half the plate thickness

Source: Roark's Formulas for Stress and Strain, 7th ed., Table 11.2, case 1a (annular plate, line load at the inner edge, outer edge simply supported, inner edge free). V3.2 reads the coefficient $K_\theta = L_9/C_7 = 0.95$ off Roark's table (that's the b/a = 0.9 value), the closed form for b/a = 0.875 is used here. C_7 is the same as in Thrust ring bending

D =^ plate flexural rigidity<br>
w_plate =^ line load along the inner edge<br>
theta_b =^ rotation of the inner edge<br>
delta_lift =^ lift of the ring at the seal

$D = \frac{E t_p^3}{12(1-\nu^2)}$<br>
$w_{plate} = \frac{F_{dyn,noz}}{2\pi r_i}$<br>
$L_9 = \frac{r_i}{r_o}\left[\frac{1+\nu}{2}\ln\frac{r_o}{r_i} + \frac{1-\nu}{4}\left(1 - \left(\frac{r_i}{r_o}\right)^2\right)\right]$<br>
$\theta_b = \frac{w_{plate} r_o^2}{D C_7} L_9$<br>
$\delta_{lift} = \theta_b w_{seal}$

Seal condition: $\delta_{lift} \le \delta_{thermal}$. NOT CHECKED YET, the thermal strain of the ring still has to be modeled

In [ ]:
D_ring = E*t_p**3/(12*(1 - nu**2)) #lbf-in, flexural rigidity
w_plate = F_dyn_nozzle/(2*math.pi*r_i) #lbf/in, line load along the inner edge
L_9 = r_i/r_o*((1 + nu)/2*math.log(r_o/r_i) + (1 - nu)/4*(1 - (r_i/r_o)**2)) #Roark Table 11.2, load at r_i
theta_b = w_plate*r_o**2*L_9/(D_ring*C_7) #radians, rotation of the inner edge
delta_lift = theta_b*w_seal #inches

print(f"Inner edge rotation: {theta_b} rad")
print(f"Lift at seal: {delta_lift} in")

# Temperature Dependence

Not used for now. Also, ill have to check which datasheet the values come from again

In [30]:
def yield_strength(T):
    """0.2% yield strength in psi at temperature T in deg C, linearly interpolated from datasheet values."""
    # [temp in C, yield strength in psi]
    temp_yield_data = np.array([[-253, 100e3],
                                [-196, 70e3],
                                [-79, 50e3],
                                [21, 35e3],
                                [205, 23e3],
                                [427, 19e3],
                                [650, 15.5e3],
                                [815, 13e3]])
    temps = temp_yield_data[:, 0]
    yields = temp_yield_data[:, 1]

    # np.interp clamps to the end values outside the data, which would overestimate strength above 815 C
    if np.any(np.asarray(T) < temps[0]) or np.any(np.asarray(T) > temps[-1]):
        raise ValueError(f"T = {T} C is outside the data range ({temps[0]:.0f} to {temps[-1]:.0f} C)")

    return np.interp(T, temps, yields)